# Advanced modeling (VLST): PR-AUC CV, early stopping, calibration, ensemble, threshold tuning

Companion to `modeling_without_time_since_stent.ipynb`. This notebook:

- Drops **`Time since stent implantation`** (same leakage rationale).
- Uses **`average_precision` (PR-AUC)** for `GridSearchCV` on LR and RF.
- Trains **CatBoost / XGBoost / LightGBM** with **validation early stopping** and a slightly wider hyperparameter search.
- **Calibrates** probabilities (sigmoid / Platt) on a held-out **calibration** slice of training data.
- **Ensembles** calibrated **Cat + XGB + RF** (and optional **TabPFN** in section **8b** + **9**): equal blend, **simplex grids** (PR-AUC / F2), **min-weight–constrained** PR, **diversity penalty** (PR − λ·max w), **K-fold mean OOF PR-AUC** on the cal slice, **no-Cat** XGB+RF lines, and **logistic stacking** on the three probabilities. With TabPFN enabled, **four-way** blends mirror that on **[Cat, XGB, RF, TabPFN]**: equal 1/4, PR / F2 grids, constrained PR, **K-fold mean OOF PR on the 4-simplex**, **diversity on the 4-simplex**, **no-Cat** optimization on the **XGB+RF+TabPFN** face, **Cat+TabPFN 50/50**, **XGB+RF+TabPFN thirds**, TabPFN-only, **PR/F2-winner + TabPFN tail** (15% and 25%), **Cat–XGB–TabPFN** / **Cat–RF–TabPFN** thirds, **XGB–TabPFN** and **RF–TabPFN** 50/50 mixes, a small **hand-set** blend (`cat_tabpfn_rf_thirds_xgb_tail`), and **4-prob stacking**. The **section 9 blend table** (and `ensemble_blend_comparison.csv`) lists **precision, recall, f1_score, accuracy**, and **`threshold_f1max_cal`** on the held-out **test** (threshold per row = F1-max on **cal**). In **section 9**, set **`ENSEMBLE_DEPLOY`**: `max_pr_cal_pool` (default) picks best cal PR-AUC in the **3-way** pool; or `nocat_f2_cal` / `stacking_lr_cal` (**deploy pool stays 3-way** so sections 10–11 behavior is unchanged).
- Chooses the **decision threshold** on the calibration slice (not 0.5): Fβ (β=2), recall at minimum precision, and optional cost.
- Evaluates once on the **frozen test set** from `preprocessing.ipynb`.

**SMOTE:** Defaults to **off** (`USE_SMOTE = False`). Synthetic oversampling often hurts PR-AUC here; keep off unless you are experimenting.

**Train/test split (see markdown at end):** Changing `test_size` in `preprocessing.ipynb` trades **training data vs. evaluation stability**; it does not reliably “improve model scores” on the same population.

## 1. Setup, load data, drop leaky column

In [4]:
import numpy as np
import pandas as pd
import os
import warnings
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.utils._tags import ClassifierTags
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
np.random.seed(42)

PROCESSED_DIR = os.path.join("..", "..", "data", "processed")
RESULT_DIR = os.path.join("..", "..", "data", "result", "modeling_advanced")
os.makedirs(RESULT_DIR, exist_ok=True)

X_train = np.load(os.path.join(PROCESSED_DIR, "X_train.npy"))
X_test = np.load(os.path.join(PROCESSED_DIR, "X_test.npy"))
y_train = np.load(os.path.join(PROCESSED_DIR, "y_train.npy"))
y_test = np.load(os.path.join(PROCESSED_DIR, "y_test.npy"))
feature_names = pd.read_csv(os.path.join(PROCESSED_DIR, "feature_names.csv"))[
    "feature_name"
].tolist()

DROP_FEATURES = ["Time since stent implantation"]
fn = np.array(feature_names)
keep = ~np.isin(fn, DROP_FEATURES)
X_train = X_train[:, keep]
X_test = X_test[:, keep]
feature_names = fn[keep].tolist()

print(f"Train {X_train.shape}, Test {X_test.shape}, features={len(feature_names)}")
print(f"Train class 0/1: {(y_train==0).sum()}/{(y_train==1).sum()}")
print(f"Test  class 0/1: {(y_test==0).sum()}/{(y_test==1).sum()}")

Train (4148, 173), Test (1037, 173), features=173
Train class 0/1: 4074/74
Test  class 0/1: 1019/18


## 1b. TabPFN API token (optional)

Run before **section 8b** if you use TabPFN. Press **Enter** to skip.

In [5]:
import getpass

_t = getpass.getpass("TabPFN API token [Enter to skip, not echoed]: ")
if str(_t).strip():
    os.environ["TABPFN_TOKEN"] = str(_t).strip()
    print("TABPFN_TOKEN set for this kernel.")
else:
    print("Skipped — export TABPFN_TOKEN in the shell or TabPFN will stay off in 8b.")

TABPFN_TOKEN set for this kernel.


## 2. Optional SMOTE (default: off)

In [6]:
from imblearn.over_sampling import SMOTE

USE_SMOTE = True

if USE_SMOTE:
    n_min = int((y_train == 1).sum())
    k = min(5, n_min - 1) if n_min > 1 else 1
    if k >= 1:
        smote = SMOTE(random_state=42, k_neighbors=k)
        X_train, y_train = smote.fit_resample(X_train, y_train)
        print("SMOTE applied:", X_train.shape, np.bincount(y_train.astype(int)))
    else:
        print("SMOTE skipped: too few minority samples.")
else:
    print("SMOTE off (recommended here based on your PR-AUC experiment).")

SMOTE applied: (8148, 173) [4074 4074]


## 3. Three-way stratified split inside training

- **fit**: model fitting (and LR/RF CV uses full `X_train` below; boosters use `X_fit`).
- **early_stop**: monitoring iterations for gradient boosting.
- **cal / threshold**: probability calibration + threshold scan (no peeking at `X_test`).

In [7]:
def three_way_stratified_split(X, y, frac_cal=0.15, frac_es=0.15, random_state=42):
    """Approximately (1 - frac_cal - frac_es) / fit, frac_es / early stop, frac_cal / calibrator."""
    X_rem, X_cal, y_rem, y_cal = train_test_split(
        X, y, test_size=frac_cal, stratify=y, random_state=random_state
    )
    es_of_rem = frac_es / (1.0 - frac_cal)
    X_fit, X_es, y_fit, y_es = train_test_split(
        X_rem, y_rem, test_size=es_of_rem, stratify=y_rem, random_state=random_state + 1
    )
    return X_fit, X_es, X_cal, y_fit, y_es, y_cal


X_fit, X_es, X_cal, y_fit, y_es, y_cal = three_way_stratified_split(
    X_train, y_train, frac_cal=0.15, frac_es=0.15, random_state=42
)
print(
    "fit:", X_fit.shape[0],
    "early_stop:", X_es.shape[0],
    "calibration/threshold:", X_cal.shape[0],
)

fit: 5702 early_stop: 1223 calibration/threshold: 1223


## 4. CatBoost wrapper + helpers

In [8]:
class CatBoostClassifierWrapper(ClassifierMixin, BaseEstimator):
    def __init__(self, **kwargs):
        self.catboost_ = CatBoostClassifier(**kwargs)

    def get_params(self, deep=True):
        return self.catboost_.get_params(deep=deep)

    def set_params(self, **params):
        self.catboost_.set_params(**params)
        return self

    def __sklearn_tags__(self):
        # Always report as classifier (sklearn is_classifier / calibration).
        tags = BaseEstimator.__sklearn_tags__(self)
        tags.estimator_type = "classifier"
        tags.classifier_tags = ClassifierTags()
        tags.target_tags.required = True
        return tags

    def fit(self, X, y, **fit_params):
        X = np.asarray(X)
        y = np.asarray(y)
        self.catboost_.fit(X, y, **fit_params)
        self.classes_ = np.unique(y)
        self.n_features_in_ = X.shape[1]
        return self

    def predict(self, X):
        return self.catboost_.predict(X)

    def predict_proba(self, X):
        return self.catboost_.predict_proba(X)


scale_pos_weight = (y_train == 0).sum() / max(int((y_train == 1).sum()), 1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING_AP = "average_precision"

## 5. Logistic regression & random forest — `GridSearchCV` with PR-AUC

In [10]:
param_grid_lr = {
    "C": [0.01, 0.1, 1.0, 10.0],
    "penalty": ["l2", "l1"],
    "solver": ["lbfgs", "liblinear"],
    "max_iter": [2000],
}
grid_lr = GridSearchCV(
    LogisticRegression(class_weight="balanced", random_state=42),
    param_grid_lr,
    scoring=SCORING_AP,
    cv=cv,
    n_jobs=-1,
    refit=True,
)
grid_lr.fit(X_train, y_train)
best_lr = grid_lr.best_estimator_
print("LR best params:", grid_lr.best_params_)
print("LR best CV average_precision:", round(grid_lr.best_score_, 4))

KeyboardInterrupt: 

In [11]:
param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [6, 10, None],
    "min_samples_leaf": [1, 2, 4],
    "class_weight": ["balanced", "balanced_subsample"],
}
grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid_rf,
    scoring=SCORING_AP,
    cv=cv,
    n_jobs=-1,
    refit=True,
)
grid_rf.fit(X_train, y_train)
best_rf = grid_rf.best_estimator_
print("RF best params:", grid_rf.best_params_)
print("RF best CV average_precision:", round(grid_rf.best_score_, 4))

/home/fadia/Documents/vlst/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/fadia/Documents/vlst/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/fadia/Documents/vlst/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/fadia/Documents/vlst/.venv/lib/python

RF best params: {'class_weight': 'balanced', 'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 100}
RF best CV average_precision: 1.0


## 6. Gradient boosting: grid + early stopping (val = `X_es`)

We maximize **average precision on `X_es`** after training on `X_fit` with early stopping.

In [12]:
def eval_ap(model, X, y):
    p = model.predict_proba(X)[:, 1]
    return average_precision_score(y, p)


def grid_search_catboost_early_stop(X_fit, y_fit, X_es, y_es):
    param_grid = {
        "depth": [4, 6, 8],
        "learning_rate": [0.03, 0.05, 0.1],
        "l2_leaf_reg": [1, 3, 5],
    }
    best = None
    best_ap = -1.0
    best_iters = 200
    for depth in param_grid["depth"]:
        for lr in param_grid["learning_rate"]:
            for l2 in param_grid["l2_leaf_reg"]:
                est = CatBoostClassifierWrapper(
                    depth=depth,
                    learning_rate=lr,
                    l2_leaf_reg=l2,
                    iterations=2000,
                    random_seed=42,
                    verbose=False,
                    auto_class_weights="Balanced",
                )
                est.fit(
                    X_fit,
                    y_fit,
                    eval_set=(X_es, y_es),
                    early_stopping_rounds=60,
                    verbose=False,
                )
                ap = eval_ap(est, X_es, y_es)
                bi = est.catboost_.get_best_iteration()
                if bi is None:
                    bi = 1999
                if ap > best_ap:
                    best_ap = ap
                    best = est
                    best_iters = int(bi) + 1
                    best_params = {"depth": depth, "learning_rate": lr, "l2_leaf_reg": l2}
    print("CatBoost best (on early-stop val):", best_params, "best_iteration~", best_iters, "AP", round(best_ap, 4))
    return best_params, best_iters


cat_params, cat_iters = grid_search_catboost_early_stop(X_fit, y_fit, X_es, y_es)
X_merge = np.vstack([X_fit, X_es])
y_merge = np.concatenate([y_fit, y_es])
best_cat = CatBoostClassifierWrapper(
    depth=cat_params["depth"],
    learning_rate=cat_params["learning_rate"],
    l2_leaf_reg=cat_params["l2_leaf_reg"],
    iterations=max(cat_iters, 50),
    random_seed=42,
    verbose=False,
    auto_class_weights="Balanced",
)
best_cat.fit(X_merge, y_merge)

CatBoost best (on early-stop val): {'depth': 8, 'learning_rate': 0.03, 'l2_leaf_reg': 1} best_iteration~ 844 AP 1.0


,iterations,844
,learning_rate,0.03
,depth,8
,l2_leaf_reg,1
,random_seed,42
,verbose,False
,auto_class_weights,'Balanced'


In [13]:
def grid_search_xgb_early_stop(X_fit, y_fit, X_es, y_es, spw):
    grid = {
        "max_depth": [4, 6, 8],
        "learning_rate": [0.03, 0.05, 0.1],
        "min_child_weight": [1, 3],
        "subsample": [0.8, 1.0],
    }
    best_ap = -1.0
    best_model = None
    best_bi = 100
    best_params = {}
    for md in grid["max_depth"]:
        for lr in grid["learning_rate"]:
            for mcw in grid["min_child_weight"]:
                for subs in grid["subsample"]:
                    m = XGBClassifier(
                        max_depth=md,
                        learning_rate=lr,
                        min_child_weight=mcw,
                        subsample=subs,
                        n_estimators=2000,
                        random_state=42,
                        scale_pos_weight=spw,
                        eval_metric="aucpr",
                        verbosity=0,
                        early_stopping_rounds=80,
                    )
                    m.fit(
                        X_fit,
                        y_fit,
                        eval_set=[(X_es, y_es)],
                        verbose=False,
                    )
                    bi = m.best_iteration
                    if bi is None:
                        bi = 0
                    ap = eval_ap(m, X_es, y_es)
                    if ap > best_ap:
                        best_ap = ap
                        best_model = m
                        best_bi = int(bi)
                        best_params = {
                            "max_depth": md,
                            "learning_rate": lr,
                            "min_child_weight": mcw,
                            "subsample": subs,
                        }
    print("XGB best (on early-stop val):", best_params, "best_iteration", best_bi, "AP", round(best_ap, 4))
    return best_params, best_bi


xgb_params, xgb_bi = grid_search_xgb_early_stop(
    X_fit, y_fit, X_es, y_es, scale_pos_weight
)
best_xgb = XGBClassifier(
    max_depth=xgb_params["max_depth"],
    learning_rate=xgb_params["learning_rate"],
    min_child_weight=xgb_params["min_child_weight"],
    subsample=xgb_params["subsample"],
    n_estimators=max(xgb_bi + 1, 50),
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    verbosity=0,
)
best_xgb.fit(X_merge, y_merge)

XGB best (on early-stop val): {'max_depth': 6, 'learning_rate': 0.03, 'min_child_weight': 3, 'subsample': 1.0} best_iteration 208 AP 1.0


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [14]:
import lightgbm as lgb


def grid_search_lgb_early_stop(X_fit, y_fit, X_es, y_es):
    grid = {
        "max_depth": [4, 6, 8],
        "learning_rate": [0.03, 0.05, 0.1],
        "min_child_samples": [5, 20, 40],
        "num_leaves": [31, 63],
    }
    best_ap = -1.0
    best_bi = 100
    best_params = {}
    for md in grid["max_depth"]:
        for lr in grid["learning_rate"]:
            for mcs in grid["min_child_samples"]:
                for nl in grid["num_leaves"]:
                    m = LGBMClassifier(
                        max_depth=md,
                        learning_rate=lr,
                        min_child_samples=mcs,
                        num_leaves=nl,
                        n_estimators=2000,
                        random_state=42,
                        class_weight="balanced",
                        verbosity=-1,
                    )
                    m.fit(
                        X_fit,
                        y_fit,
                        eval_set=[(X_es, y_es)],
                        eval_metric="auc",
                        callbacks=[
                            lgb.log_evaluation(period=0),
                            lgb.early_stopping(
                                stopping_rounds=60,
                                first_metric_only=True,
                                verbose=False,
                            ),
                        ],
                    )
                    bi = m.best_iteration_
                    if bi is None:
                        bi = 0
                    ap = eval_ap(m, X_es, y_es)
                    if ap > best_ap:
                        best_ap = ap
                        best_bi = int(bi)
                        best_params = {
                            "max_depth": md,
                            "learning_rate": lr,
                            "min_child_samples": mcs,
                            "num_leaves": nl,
                        }
    print(
        "LGB best (on early-stop val):",
        best_params,
        "best_iteration",
        best_bi,
        "AP",
        round(best_ap, 4),
    )
    return best_params, best_bi


lgb_params, lgb_bi = grid_search_lgb_early_stop(X_fit, y_fit, X_es, y_es)
best_lgb = LGBMClassifier(
    max_depth=lgb_params["max_depth"],
    learning_rate=lgb_params["learning_rate"],
    min_child_samples=lgb_params["min_child_samples"],
    num_leaves=lgb_params["num_leaves"],
    n_estimators=max(lgb_bi + 1, 50),
    random_state=42,
    class_weight="balanced",
    verbosity=-1,
)
best_lgb.fit(X_merge, y_merge)

LGB best (on early-stop val): {'max_depth': 4, 'learning_rate': 0.05, 'min_child_samples': 5, 'num_leaves': 31} best_iteration 219 AP 1.0


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,4
,learning_rate,0.05
,n_estimators,220
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,5


## 7. Refit RF on merged fit+ES (align with boosters)

Grid RF was trained on full `X_train`; for the ensemble we use the same **merge** as boosters so calibration sees consistent behavior. Optional: refit RF on `X_merge` with best params.

In [15]:
best_rf_merge = clone(best_rf)
best_rf_merge.fit(X_merge, y_merge)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

## 8. Sigmoid calibration on `X_cal` (prefit)

In [16]:
def calibrate_prefit(base_fitted, X_cal, y_cal, method="sigmoid"):
    """Post-hoc calibration for an already-fitted classifier.

    sklearn 1.8 removed ``cv='prefit'``. Wrap the base model in
    :class:`~sklearn.frozen.FrozenEstimator` so it is not re-trained;
    internal CV only obtains out-of-fold scores for the calibrator.
    """
    n_pos = int((y_cal == 1).sum())
    n_neg = int((y_cal == 0).sum())
    k = min(5, n_pos, n_neg)
    if k < 2:
        raise ValueError(
            f"Calibration needs >=2 samples per class in X_cal; got pos={n_pos}, neg={n_neg}"
        )
    Xc = np.asarray(X_cal)
    if not hasattr(base_fitted, "classes_"):
        base_fitted.classes_ = np.unique(y_cal)
    if not hasattr(base_fitted, "n_features_in_"):
        base_fitted.n_features_in_ = Xc.shape[1]
    cal = CalibratedClassifierCV(
        estimator=FrozenEstimator(base_fitted),
        method=method,
        cv=k,
    )
    cal.fit(X_cal, y_cal)
    return cal


cal_cat = calibrate_prefit(best_cat, X_cal, y_cal, method="sigmoid")
cal_xgb = calibrate_prefit(best_xgb, X_cal, y_cal, method="sigmoid")
cal_rf = calibrate_prefit(best_rf_merge, X_cal, y_cal, method="sigmoid")
print("Calibrated CatBoost, XGB, RF on calibration slice.")

Calibrated CatBoost, XGB, RF on calibration slice.


## 8b. TabPFN (optional) for four-way blends in section 9

- **Default:** `RUN_TABPFN_BLEND = False` so a long TabPFN fit never starts unexpectedly (and does not compete with another TabPFN notebook/kernel).
- **Enable:** set `RUN_TABPFN_BLEND = True`, ensure **`TABPFN_TOKEN`** is exported (same as `tabpfn.ipynb`), then run this cell before section 9.
- TabPFN is **fit on `X_merge`** (same as boosters / RF merge), then **sigmoid-calibrated** on `X_cal` like the other ensemble members. Section 9 then adds four-way grids, **mean OOF PR** on the 4-simplex, TabPFN tails (15% / 25%), thirds / 50–50 diagnostics, and extra rows in `ensemble_blend_comparison.csv` (with **precision, recall, f1_score, accuracy** on test).

In [ ]:
# Optional TabPFN for §9 four-way blends (off by default).
RUN_TABPFN_BLEND = True
TABPFN_BALANCE_PROBABILITIES = True
TABPFN_N_ESTIMATORS = 8
TABPFN_IGNORE_PRETRAINING_LIMITS = True
TABPFN_DEVICE = "auto"

TABPFN_BLEND_AVAILABLE = False
cal_tabpfn = None
p_tabpfn_cal = None
p_tabpfn_test = None

if RUN_TABPFN_BLEND:
    if not os.environ.get("TABPFN_TOKEN"):
        print(
            "RUN_TABPFN_BLEND is True but TABPFN_TOKEN is not set; skipping TabPFN (export token like tabpfn.ipynb)."
        )
    else:
        from tabpfn import TabPFNClassifier

        _tabpfn_base = TabPFNClassifier(
            random_state=42,
            n_estimators=TABPFN_N_ESTIMATORS,
            device=TABPFN_DEVICE,
            ignore_pretraining_limits=TABPFN_IGNORE_PRETRAINING_LIMITS,
            balance_probabilities=TABPFN_BALANCE_PROBABILITIES,
        )
        _tabpfn_base.fit(X_merge, y_merge)
        cal_tabpfn = calibrate_prefit(_tabpfn_base, X_cal, y_cal, method="sigmoid")
        p_tabpfn_cal = cal_tabpfn.predict_proba(X_cal)[:, 1]
        p_tabpfn_test = cal_tabpfn.predict_proba(X_test)[:, 1]
        TABPFN_BLEND_AVAILABLE = True
        print(
            "TabPFN fit + sigmoid cal OK | PR-AUC cal",
            round(average_precision_score(y_cal, p_tabpfn_cal), 4),
            "| test",
            round(average_precision_score(y_test, p_tabpfn_test), 4),
        )
else:
    print("TabPFN blends: OFF (set RUN_TABPFN_BLEND = True after §8 to add four-way mixes in §9).")

## 9. Ensemble blends: grids, constraints, CV on cal, diversity penalty, stacking

The blend **table** shows **PR-AUC / ROC-AUC** (threshold-free), then **precision, recall, f1_score, accuracy**, and **`threshold_f1max_cal`** on the **test** set (per-row threshold = **F1-max on the calibration** slice). When **section 8b** is enabled, extra **TabPFN-inclusive** rows appear: four-way grids, **mean OOF PR** on the 4-simplex, no-Cat face, diversity, **15% / 25% TabPFN tails** on 3-way PR/F2 winners, pairwise and one-third diagnostic mixes, and **stacking** on four probabilities.

In [16]:
def stack_proba(models, X):
    return np.column_stack([m.predict_proba(X)[:, 1] for m in models])


def blend(P, w):
    w = np.asarray(w, dtype=float)
    w = w / w.sum()
    return (P @ w).ravel()


def simplex_grid_3(step=0.05):
    rows = []
    s = np.arange(0, 1 + step * 0.5, step)
    for a in s:
        for b in s:
            c = 1.0 - a - b
            if c >= -1e-9:
                rows.append([a, b, max(c, 0.0)])
    return np.array(rows)


def max_f2_over_thresholds(y_true, p, t_grid):
    best = 0.0
    for t in t_grid:
        y_hat = (p >= t).astype(int)
        best = max(best, fbeta_score(y_true, y_hat, beta=2, zero_division=0))
    return best


# --- tuning knobs (calibration only; adjust and re-run from §9) ---
TABPFN_BLEND_AVAILABLE = bool(globals().get("TABPFN_BLEND_AVAILABLE", False))
BLEND_MIN_WEIGHT = 0.10  # constrained convex blend: each base model at least this share
BLEND_DIVERSITY_LAMBDA = 0.08  # maximize PR-AUC_cal - lambda * max(w) on the simplex grid
BLEND_CV_FOLDS = 5  # stratified folds on X_cal for mean out-of-fold PR-AUC
# Deploy: pick the candidate below with highest PR-AUC on full X_cal (no test).
# Options: "max_pr_cal_pool" | "nocat_f2_cal" | "stacking_lr_cal"
ENSEMBLE_DEPLOY = "max_pr_cal_pool"

ens_models = [cal_cat, cal_xgb, cal_rf]
P_cal = stack_proba(ens_models, X_cal)
P_test = stack_proba(ens_models, X_test)

t_grid_blend = np.arange(0.05, 0.96, 0.01)


def best_threshold_f1_on_cal(y_cal_arr, p_cal_arr, grid):
    best_t, best_f1 = 0.5, -1.0
    for t in grid:
        y_hat = (p_cal_arr >= t).astype(int)
        fv = f1_score(y_cal_arr, y_hat, zero_division=0)
        if fv > best_f1:
            best_f1, best_t = fv, float(t)
    return best_t


def test_metrics_at_threshold(y_te, p_te, t):
    y_hat = (p_te >= t).astype(int)
    return {
        "precision": precision_score(y_te, y_hat, zero_division=0),
        "recall": recall_score(y_te, y_hat, zero_division=0),
        "f1_score": f1_score(y_te, y_hat, zero_division=0),
        "accuracy": accuracy_score(y_te, y_hat),
        "threshold_f1max_cal": float(t),
    }


W3 = simplex_grid_3(0.05)
W3_constrained = W3[np.all(W3 >= BLEND_MIN_WEIGHT - 1e-9, axis=1)]
if len(W3_constrained) == 0:
    W3_constrained = W3

w_equal = np.array([1 / 3, 1 / 3, 1 / 3])
best_ap, w_ap = -1.0, w_equal.copy()
best_f2c, w_f2 = -1.0, w_equal.copy()
for w in W3:
    p = blend(P_cal, w)
    ap = average_precision_score(y_cal, p)
    if ap > best_ap:
        best_ap, w_ap = ap, w.copy()
    f2c = max_f2_over_thresholds(y_cal, p, t_grid_blend)
    if f2c > best_f2c:
        best_f2c, w_f2 = f2c, w.copy()

best_ap_c, w_ap_constrained = -1.0, w_equal.copy()
for w in W3_constrained:
    ap = average_precision_score(y_cal, blend(P_cal, w))
    if ap > best_ap_c:
        best_ap_c, w_ap_constrained = ap, w.copy()

best_div, w_div = -1e18, w_equal.copy()
for w in W3:
    p = blend(P_cal, w)
    ap = average_precision_score(y_cal, p)
    sc = ap - BLEND_DIVERSITY_LAMBDA * float(np.max(w))
    if sc > best_div:
        best_div, w_div = sc, w.copy()

cv_fold_Py = []
skf_blend = StratifiedKFold(n_splits=BLEND_CV_FOLDS, shuffle=True, random_state=42)
for _, val_idx in skf_blend.split(X_cal, y_cal):
    P_v = stack_proba(ens_models, X_cal[val_idx])
    y_v = y_cal[val_idx]
    cv_fold_Py.append((P_v, y_v))


def mean_oof_pr_auc(w):
    return float(
        np.mean(
            [average_precision_score(yv, blend(Pv, w)) for Pv, yv in cv_fold_Py]
        )
    )


best_cv, w_cv = -1.0, w_equal.copy()
for w in W3:
    m = mean_oof_pr_auc(w)
    if m > best_cv:
        best_cv, w_cv = m, w.copy()

best_ap_2, w_xgb_rf = -1.0, np.array([0.5, 0.5])
best_f2_2, w_xgb_rf_f2 = -1.0, np.array([0.5, 0.5])
for wx in np.arange(0, 1.0001, 0.05):
    wr = 1.0 - wx
    w = np.array([0.0, wx, wr])
    p = blend(P_cal, w)
    ap = average_precision_score(y_cal, p)
    if ap > best_ap_2:
        best_ap_2, w_xgb_rf = ap, np.array([wx, wr])
    f2c = max_f2_over_thresholds(y_cal, p, t_grid_blend)
    if f2c > best_f2_2:
        best_f2_2, w_xgb_rf_f2 = f2c, np.array([wx, wr])

w_nocat_ap = np.array([0.0, w_xgb_rf[0], w_xgb_rf[1]])
w_nocat_f2 = np.array([0.0, w_xgb_rf_f2[0], w_xgb_rf_f2[1]])

stack_lr = LogisticRegression(
    max_iter=2000, class_weight="balanced", random_state=42, solver="lbfgs"
)
stack_lr.fit(P_cal, y_cal)
p_cal_stack = stack_lr.predict_proba(P_cal)[:, 1]
p_test_stack = stack_lr.predict_proba(P_test)[:, 1]


def simplex_grid_4(step=0.1):
    rows = []
    s = np.arange(0, 1 + step * 0.5, step)
    for a in s:
        for b in s:
            for c in s:
                d = 1.0 - a - b - c
                if d >= -1e-9:
                    rows.append([a, b, c, max(d, 0.0)])
    return np.array(rows)


def row_variant_4(name, w):
    w = np.asarray(w, dtype=float)
    p_c = blend(P_cal_4, w)
    p_te = blend(P_test_4, w)
    t_opt = best_threshold_f1_on_cal(y_cal, p_c, t_grid_blend)
    row = {
        "variant": name,
        "w_cat": w[0],
        "w_xgb": w[1],
        "w_rf": w[2],
        "w_tabpfn": w[3],
        "pr_auc_cal": average_precision_score(y_cal, p_c),
        "max_f2_cal": max_f2_over_thresholds(y_cal, p_c, t_grid_blend),
        "pr_auc_test": average_precision_score(y_test, p_te),
        "roc_auc_test": roc_auc_score(y_test, p_te),
    }
    row.update(test_metrics_at_threshold(y_test, p_te, t_opt))
    return row


if TABPFN_BLEND_AVAILABLE:
    P_cal_4 = np.column_stack([P_cal, p_tabpfn_cal])
    P_test_4 = np.column_stack([P_test, p_tabpfn_test])
    W4 = simplex_grid_4(0.1)
    W4_constrained = W4[np.all(W4 >= BLEND_MIN_WEIGHT - 1e-9, axis=1)]
    if len(W4_constrained) == 0:
        W4_constrained = W4
    w4_equal = np.array([0.25, 0.25, 0.25, 0.25])
    best_ap4, w_ap4 = -1.0, w4_equal.copy()
    best_f24, w_f24 = -1.0, w4_equal.copy()
    for w in W4:
        p = blend(P_cal_4, w)
        ap = average_precision_score(y_cal, p)
        if ap > best_ap4:
            best_ap4, w_ap4 = ap, w.copy()
        f2c = max_f2_over_thresholds(y_cal, p, t_grid_blend)
        if f2c > best_f24:
            best_f24, w_f24 = f2c, w.copy()
    best_ap4c, w_ap4_constrained = -1.0, w4_equal.copy()
    for w in W4_constrained:
        ap = average_precision_score(y_cal, blend(P_cal_4, w))
        if ap > best_ap4c:
            best_ap4c, w_ap4_constrained = ap, w.copy()
    stack_lr_4 = LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=43, solver="lbfgs"
    )
    stack_lr_4.fit(P_cal_4, y_cal)
    p_cal_stack4 = stack_lr_4.predict_proba(P_cal_4)[:, 1]
    p_test_stack4 = stack_lr_4.predict_proba(P_test_4)[:, 1]
    best_div4, w_div4 = -1e18, w4_equal.copy()
    for w in W4:
        p = blend(P_cal_4, w)
        ap = average_precision_score(y_cal, p)
        sc = ap - BLEND_DIVERSITY_LAMBDA * float(np.max(w))
        if sc > best_div4:
            best_div4, w_div4 = sc, w.copy()
    W_nocat4 = np.concatenate([np.zeros((len(W3), 1)), W3], axis=1)
    best_nc_ap, w_nocat_ap4 = -1.0, W_nocat4[0].copy()
    best_nc_f2, w_nocat_f24 = -1.0, W_nocat4[0].copy()
    for w in W_nocat4:
        p = blend(P_cal_4, w)
        ap = average_precision_score(y_cal, p)
        if ap > best_nc_ap:
            best_nc_ap, w_nocat_ap4 = ap, w.copy()
        f2c = max_f2_over_thresholds(y_cal, p, t_grid_blend)
        if f2c > best_nc_f2:
            best_nc_f2, w_nocat_f24 = f2c, w.copy()
    # Same OOF idea as 3-way w_cv: cal-fold splits on X_cal, TabPFN column from fixed cal_tabpfn.
    cv_fold_Py_4 = []
    for _, val_idx in skf_blend.split(X_cal, y_cal):
        P_v3 = stack_proba(ens_models, X_cal[val_idx])
        P_v4 = np.column_stack(
            [P_v3, cal_tabpfn.predict_proba(X_cal[val_idx])[:, 1]]
        )
        cv_fold_Py_4.append((P_v4, y_cal[val_idx]))

    def mean_oof_pr_auc4(w):
        return float(
            np.mean(
                [
                    average_precision_score(yv, blend(Pv, w))
                    for Pv, yv in cv_fold_Py_4
                ]
            )
        )

    best_cv4, w_cv4 = -1.0, w4_equal.copy()
    for w in W4:
        m4 = mean_oof_pr_auc4(w)
        if m4 > best_cv4:
            best_cv4, w_cv4 = m4, w.copy()
else:
    P_cal_4 = P_test_4 = None
    w_ap4 = w_f24 = w_ap4_constrained = w4_equal = None
    w_div4 = w_nocat_ap4 = w_nocat_f24 = None
    w_cv4 = None
    p_cal_stack4 = p_test_stack4 = None


def row_variant(name, w):
    p_c = blend(P_cal, w)
    p_te = blend(P_test, w)
    t_opt = best_threshold_f1_on_cal(y_cal, p_c, t_grid_blend)
    row = {
        "variant": name,
        "w_cat": w[0],
        "w_xgb": w[1],
        "w_rf": w[2],
        "w_tabpfn": float("nan"),
        "pr_auc_cal": average_precision_score(y_cal, p_c),
        "max_f2_cal": max_f2_over_thresholds(y_cal, p_c, t_grid_blend),
        "pr_auc_test": average_precision_score(y_test, p_te),
        "roc_auc_test": roc_auc_score(y_test, p_te),
    }
    row.update(test_metrics_at_threshold(y_test, p_te, t_opt))
    return row


def row_from_probs(name, p_c, p_te):
    t_opt = best_threshold_f1_on_cal(y_cal, p_c, t_grid_blend)
    row = {
        "variant": name,
        "w_cat": float("nan"),
        "w_xgb": float("nan"),
        "w_rf": float("nan"),
        "w_tabpfn": float("nan"),
        "pr_auc_cal": average_precision_score(y_cal, p_c),
        "max_f2_cal": max_f2_over_thresholds(y_cal, p_c, t_grid_blend),
        "pr_auc_test": average_precision_score(y_test, p_te),
        "roc_auc_test": roc_auc_score(y_test, p_te),
    }
    row.update(test_metrics_at_threshold(y_test, p_te, t_opt))
    return row


blend_rows = [
    row_variant("equal_1/3", w_equal),
    row_variant("grid_max_pr_auc_cal", w_ap),
    row_variant(
        f"grid_max_pr_auc_cal_min_w{int(round(BLEND_MIN_WEIGHT * 100))}pct",
        w_ap_constrained,
    ),
    row_variant(
        f"grid_pr_minus_{BLEND_DIVERSITY_LAMBDA:g}x_maxw",
        w_div,
    ),
    row_variant(f"grid_mean_{BLEND_CV_FOLDS}fold_oof_pr_cal", w_cv),
    row_variant("grid_max_f2_cal", w_f2),
    row_variant("nocat_max_pr_auc_cal", w_nocat_ap),
    row_variant("nocat_max_f2_cal", w_nocat_f2),
    row_from_probs("stacking_lr_on_probs_cal", p_cal_stack, p_test_stack),
]
if TABPFN_BLEND_AVAILABLE:
    blend_rows += [
        row_variant_4("four_equal_1/4", w4_equal),
        row_variant_4("four_grid_max_pr_auc_cal", w_ap4),
        row_variant_4(
            f"four_grid_mean_{BLEND_CV_FOLDS}fold_oof_pr_cal",
            w_cv4,
        ),
        row_variant_4(
            f"four_grid_max_pr_min_w{int(round(BLEND_MIN_WEIGHT * 100))}pct",
            w_ap4_constrained,
        ),
        row_variant_4("four_grid_max_f2_cal", w_f24),
        row_variant_4("tabpfn_only_calibrated", np.array([0.0, 0.0, 0.0, 1.0])),
        row_variant_4(
            "three_way_pr_winner_plus_tabpfn_15pct",
            np.array(
                [
                    0.85 * w_ap[0],
                    0.85 * w_ap[1],
                    0.85 * w_ap[2],
                    0.15,
                ]
            ),
        ),
        row_variant_4(
            "three_way_f2_winner_plus_tabpfn_15pct",
            np.array(
                [
                    0.85 * w_f2[0],
                    0.85 * w_f2[1],
                    0.85 * w_f2[2],
                    0.15,
                ]
            ),
        ),
        row_variant_4(
            f"four_grid_pr_minus_{BLEND_DIVERSITY_LAMBDA:g}x_maxw",
            w_div4,
        ),
        row_variant_4("four_nocat_max_pr_auc_cal", w_nocat_ap4),
        row_variant_4("four_nocat_max_f2_cal", w_nocat_f24),
        row_variant_4("cat_tabpfn_50_50", np.array([0.5, 0.0, 0.0, 0.5])),
        row_variant_4(
            "xgb_rf_tabpfn_equal_thirds",
            np.array([0.0, 1.0 / 3, 1.0 / 3, 1.0 / 3]),
        ),
        row_variant_4(
            "three_way_pr_winner_plus_tabpfn_25pct",
            np.array(
                [
                    0.75 * w_ap[0],
                    0.75 * w_ap[1],
                    0.75 * w_ap[2],
                    0.25,
                ]
            ),
        ),
        row_variant_4(
            "three_way_f2_winner_plus_tabpfn_25pct",
            np.array(
                [
                    0.75 * w_f2[0],
                    0.75 * w_f2[1],
                    0.75 * w_f2[2],
                    0.25,
                ]
            ),
        ),
        row_variant_4(
            "cat_xgb_tabpfn_equal_thirds",
            np.array([1.0 / 3, 1.0 / 3, 0.0, 1.0 / 3]),
        ),
        row_variant_4(
            "cat_rf_tabpfn_equal_thirds",
            np.array([1.0 / 3, 0.0, 1.0 / 3, 1.0 / 3]),
        ),
        row_variant_4(
            "xgb_tabpfn_50_50",
            np.array([0.0, 0.5, 0.0, 0.5]),
        ),
        row_variant_4(
            "rf_tabpfn_50_50",
            np.array([0.0, 0.0, 0.5, 0.5]),
        ),
        row_variant_4(
            "cat_tabpfn_rf_thirds_xgb_tail",
            np.array([0.30, 0.10, 0.30, 0.30]),
        ),
        row_from_probs("four_stacking_lr_on_probs_cal", p_cal_stack4, p_test_stack4),
    ]
blend_table = pd.DataFrame(blend_rows)
_blend_display_cols = [
    "variant",
    "w_cat",
    "w_xgb",
    "w_rf",
    "w_tabpfn",
    "pr_auc_cal",
    "max_f2_cal",
    "pr_auc_test",
    "roc_auc_test",
    "precision",
    "recall",
    "f1_score",
    "accuracy",
    "threshold_f1max_cal",
]
display(
    blend_table[[c for c in _blend_display_cols if c in blend_table.columns]].round(4)
)
blend_table.to_csv(os.path.join(RESULT_DIR, "ensemble_blend_comparison.csv"), index=False)

pool_pr_cal = {
    "equal_1/3": (blend(P_cal, w_equal), blend(P_test, w_equal)),
    "grid_max_pr_auc_cal": (blend(P_cal, w_ap), blend(P_test, w_ap)),
    "nocat_max_pr_auc_cal": (blend(P_cal, w_nocat_ap), blend(P_test, w_nocat_ap)),
    f"grid_max_pr_min_w{int(round(BLEND_MIN_WEIGHT * 100))}pct": (
        blend(P_cal, w_ap_constrained),
        blend(P_test, w_ap_constrained),
    ),
    f"grid_pr_minus_{BLEND_DIVERSITY_LAMBDA:g}x_maxw": (
        blend(P_cal, w_div),
        blend(P_test, w_div),
    ),
    f"mean_{BLEND_CV_FOLDS}fold_oof_pr": (
        blend(P_cal, w_cv),
        blend(P_test, w_cv),
    ),
    "nocat_max_f2_cal": (blend(P_cal, w_nocat_f2), blend(P_test, w_nocat_f2)),
    "stacking_lr_cal": (p_cal_stack, p_test_stack),
}

if ENSEMBLE_DEPLOY == "max_pr_cal_pool":
    win_name = max(
        pool_pr_cal,
        key=lambda k: average_precision_score(y_cal, pool_pr_cal[k][0]),
    )
    p_cal, p_test = pool_pr_cal[win_name]
elif ENSEMBLE_DEPLOY == "nocat_f2_cal":
    win_name = "nocat_max_f2_cal"
    p_cal, p_test = pool_pr_cal[win_name]
elif ENSEMBLE_DEPLOY == "stacking_lr_cal":
    win_name = "stacking_lr_cal"
    p_cal, p_test = pool_pr_cal[win_name]
else:
    raise ValueError(f"Unknown ENSEMBLE_DEPLOY={ENSEMBLE_DEPLOY!r}")

p_test_equal = blend(P_test, w_equal)
p_test_unc_pr = blend(P_test, w_ap)
pr_test_equal = average_precision_score(y_test, p_test_equal)
pr_test_unc = average_precision_score(y_test, p_test_unc_pr)
print(
    "Tradeoff (test — for diagnostics only): unconstrained cal-PR grid winner vs equal blend —",
    "test PR-AUC",
    round(pr_test_unc, 4),
    "vs",
    round(pr_test_equal, 4),
)
print(
    "Downstream sections 10–11 deploy:",
    ENSEMBLE_DEPLOY,
    "→",
    win_name,
    "| cal PR-AUC",
    round(average_precision_score(y_cal, p_cal), 4),
    "| test PR-AUC",
    round(average_precision_score(y_test, p_test), 4),
    "| test ROC",
    round(roc_auc_score(y_test, p_test), 4),
)
if win_name == "stacking_lr_cal":
    print("Stacking LR coef (cat, xgb, rf):", np.round(stack_lr.coef_.ravel(), 4).tolist())


,variant,w_cat,w_xgb,w_rf,w_tabpfn,pr_auc_cal,max_f2_cal,pr_auc_test,roc_auc_test,precision,recall,f1_score,accuracy,threshold_f1max_cal
0,equal_1/3,0.3333,0.3333,0.3333,NaN,0.6997,0.6731,0.6795,0.9547,0.6471,0.6111,0.6286,0.9875,0.28
1,grid_max_pr_auc_cal,0.0000,1.0000,0.0000,NaN,0.7773,0.6863,0.6600,0.9713,0.6154,0.4444,0.5161,0.9855,0.11
2,grid_max_pr_auc_cal_min_w10pct,0.1000,0.8000,0.1000,NaN,0.7396,0.6863,0.6479,0.9560,0.6667,0.4444,0.5333,0.9865,0.16
3,grid_pr_minus_0.08x_maxw,0.0000,1.0000,0.0000,NaN,0.7773,0.6863,0.6600,0.9713,0.6154,0.4444,0.5161,0.9855,0.11
4,grid_mean_5fold_oof_pr_cal,0.0000,1.0000,0.0000,NaN,0.7773,0.6863,0.6600,0.9713,0.6154,0.4444,0.5161,0.9855,0.11
5,grid_max_f2_cal,0.1500,0.8500,0.0000,NaN,0.7302,0.6897,0.6330,0.9629,0.6154,0.4444,0.5161,0.9855,0.14
6,nocat_max_pr_auc_cal,0.0000,1.0000,0.0000,NaN,0.7773,0.6863,0.6600,0.9713,0.6154,0.4444,0.5161,0.9855,0.11
7,nocat_max_f2_cal,0.0000,0.6000,0.4000,NaN,0.7210,0.6863,0.7039,0.9503,1.0000,0.4444,0.6154,0.9904,0.41
8,stacking_lr_on_probs_cal,NaN,NaN,NaN,NaN,0.7259,0.6863,0.6588,0.9565,1.0000,0.3889,0.5600,0.9894,0.94


Tradeoff (test — for diagnostics only): unconstrained cal-PR grid winner vs equal blend — test PR-AUC 0.66 vs 0.6795
Downstream sections 10–11 deploy: max_pr_cal_pool → grid_max_pr_auc_cal | cal PR-AUC 0.7773 | test PR-AUC 0.66 | test ROC 0.9713


## 10. Threshold selection on **calibration** data (not test)

Objectives:
- **F2** (β=2): weights recall higher than F1.
- **Recall at precision ≥** a floor (here 0.35; adjust to your clinical floor).
- **Expected cost**: `cost = C_FP * FP + C_FN * FN` (tune weights).

In [17]:
def metrics_at_threshold(y_true, p, t):
    y_hat = (p >= t).astype(int)
    return {
        "threshold": t,
        "precision": precision_score(y_true, y_hat, zero_division=0),
        "recall": recall_score(y_true, y_hat, zero_division=0),
        "f1": f1_score(y_true, y_hat, zero_division=0),
        "f2": fbeta_score(y_true, y_hat, beta=2, zero_division=0),
    }


def recall_at_min_precision(y_true, p, p_min=0.35):
    best_r = 0.0
    best_t = 0.5
    for t in np.linspace(0.01, 0.99, 199):
        row = metrics_at_threshold(y_true, p, t)
        if row["precision"] >= p_min:
            if row["recall"] > best_r:
                best_r = row["recall"]
                best_t = t
    return best_t, best_r


def best_cost_threshold(y_true, p, c_fp=1.0, c_fn=10.0):
    """Lower C_FN relative to C_FP if missing a case is worse than a false alarm."""
    best = None
    best_cost = 1e18
    n = len(y_true)
    pos = int(y_true.sum())
    neg = n - pos
    for t in np.linspace(0.01, 0.99, 199):
        y_hat = (p >= t).astype(int)
        fp = int(((y_hat == 1) & (y_true == 0)).sum())
        fn = int(((y_hat == 0) & (y_true == 1)).sum())
        cost = c_fp * fp + c_fn * fn
        if cost < best_cost:
            best_cost = cost
            best = t
    return best, best_cost


thresholds_grid = np.arange(0.05, 0.96, 0.01)
rows = [metrics_at_threshold(y_cal, p_cal, t) for t in thresholds_grid]
threshold_scan_cal = pd.DataFrame(rows)

t_f2 = float(threshold_scan_cal.loc[threshold_scan_cal["f2"].idxmax(), "threshold"])
t_rp, r_at_p = recall_at_min_precision(y_cal, p_cal, p_min=0.35)
t_cost, total_cost = best_cost_threshold(y_cal, p_cal, c_fp=1.0, c_fn=15.0)

print("Chosen on calibration — max F2 threshold:", round(t_f2, 3))
print("Chosen on calibration — recall@P>=0.35: t =", round(t_rp, 3), "recall", round(r_at_p, 3))
print("Chosen on calibration — min cost (C_FP=1,C_FN=15): t =", round(t_cost, 3), "cost", total_cost)

THRESHOLD_DEPLOY = t_f2
print("\nDeploy threshold (edit THRESHOLD_DEPLOY if you prefer another rule):", THRESHOLD_DEPLOY)

Chosen on calibration — max F2 threshold: 0.11
Chosen on calibration — recall@P>=0.35: t = 0.01 recall 0.818
Chosen on calibration — min cost (C_FP=1,C_FN=15): t = 0.01 cost 43.0

Deploy threshold (edit THRESHOLD_DEPLOY if you prefer another rule): 0.11000000000000001


## 11. Test evaluation: 0.5 vs tuned threshold

In [18]:
def report_at_t(name, y_te, p_te, t):
    y_hat = (p_te >= t).astype(int)
    print(f"\n=== {name} @ threshold={t:.3f} ===")
    print(classification_report(y_te, y_hat, digits=3))
    print("ROC-AUC:", round(roc_auc_score(y_te, p_te), 4), "| PR-AUC:", round(average_precision_score(y_te, p_te), 4))


report_at_t(f"Ensemble (deploy={win_name})", y_test, p_test, 0.5)
report_at_t(f"Ensemble (deploy={win_name})", y_test, p_test, THRESHOLD_DEPLOY)

threshold_scan_cal.to_csv(
    os.path.join(RESULT_DIR, "threshold_scan_calibration.csv"), index=False
)
summary = pd.DataFrame(
    [
        {
            "rule": "max_F2_on_cal",
            "threshold": t_f2,
            "note": "deploy default",
        },
        {
            "rule": "recall_at_precision_ge_0.35",
            "threshold": t_rp,
            "max_recall_under_constraint": r_at_p,
        },
        {
            "rule": "min_cost_fp1_fn15",
            "threshold": t_cost,
            "total_cost_on_cal": total_cost,
        },
    ]
)
summary.to_csv(os.path.join(RESULT_DIR, "threshold_rules_summary.csv"), index=False)
print("\nSaved:", RESULT_DIR)


=== Ensemble (deploy=grid_max_pr_auc_cal) @ threshold=0.500 ===
              precision    recall  f1-score   support

           0      0.989     1.000     0.995      1019
           1      1.000     0.389     0.560        18

    accuracy                          0.989      1037
   macro avg      0.995     0.694     0.777      1037
weighted avg      0.990     0.989     0.987      1037

ROC-AUC: 0.9713 | PR-AUC: 0.66

=== Ensemble (deploy=grid_max_pr_auc_cal) @ threshold=0.110 ===
              precision    recall  f1-score   support

           0      0.990     0.995     0.993      1019
           1      0.615     0.444     0.516        18

    accuracy                          0.986      1037
   macro avg      0.803     0.720     0.754      1037
weighted avg      0.984     0.986     0.984      1037

ROC-AUC: 0.9713 | PR-AUC: 0.66

Saved: ../../data/result/modeling_advanced


## 12. Quick benchmark: individual calibrated models on test (probabilities)

Columns **precision**, **recall**, **f1**, **accuracy** use **`THRESHOLD_DEPLOY`** from §10 (same rule for every row so the table is comparable). **roc_auc** / **pr_auc** are threshold-free.

In [19]:
def bench_row(name, pt, t_deploy):
    y_hat = (pt >= t_deploy).astype(int)
    return {
        "model": name,
        "roc_auc": roc_auc_score(y_test, pt),
        "pr_auc": average_precision_score(y_test, pt),
        "precision": precision_score(y_test, y_hat, zero_division=0),
        "recall": recall_score(y_test, y_hat, zero_division=0),
        "f1": f1_score(y_test, y_hat, zero_division=0),
        "accuracy": accuracy_score(y_test, y_hat),
        "threshold_deploy": float(t_deploy),
    }


_tdep = float(THRESHOLD_DEPLOY)
rows = []
for name, m in [("LR", best_lr), ("RF_merge", best_rf_merge), ("Cat_cal", cal_cat), ("XGB_cal", cal_xgb), ("LGB", best_lgb)]:
    rows.append(bench_row(name, m.predict_proba(X_test)[:, 1], _tdep))
rows.append(bench_row("Ensemble_avg_cal", blend(P_test, w_equal), _tdep))
rows.append(bench_row("Ensemble_w_pr_full_cal", blend(P_test, w_ap), _tdep))
rows.append(
    bench_row(
        f"Ensemble_w_pr_constrained_min{int(round(BLEND_MIN_WEIGHT * 100))}pct_cal",
        blend(P_test, w_ap_constrained),
        _tdep,
    )
)
rows.append(
    bench_row(
        f"Ensemble_w_pr_divpen_{BLEND_DIVERSITY_LAMBDA:g}_cal",
        blend(P_test, w_div),
        _tdep,
    )
)
rows.append(
    bench_row(
        f"Ensemble_w_mean{BLEND_CV_FOLDS}fold_oof_pr_cal",
        blend(P_test, w_cv),
        _tdep,
    )
)
rows.append(bench_row("Ensemble_w_f2_full_cal", blend(P_test, w_f2), _tdep))
rows.append(bench_row("Ensemble_nocat_w_pr_cal", blend(P_test, w_nocat_ap), _tdep))
rows.append(bench_row("Ensemble_nocat_w_f2_cal", blend(P_test, w_nocat_f2), _tdep))
rows.append(bench_row("Ensemble_stacking_lr_cal", p_test_stack, _tdep))
rows.append(bench_row("Ensemble_used_in_sec10_11", p_test, _tdep))
_tabpfn_bench_ok = (
    globals().get("TABPFN_BLEND_AVAILABLE", False)
    and globals().get("p_tabpfn_test") is not None
    and globals().get("w4_equal") is not None
    and globals().get("P_test_4") is not None
)
if _tabpfn_bench_ok:
    rows.append(bench_row("TabPFN_cal", p_tabpfn_test, _tdep))
    rows.append(bench_row("four_equal_1/4_tabpfn_mix", blend(P_test_4, w4_equal), _tdep))
    rows.append(bench_row("four_grid_max_pr_auc_cal", blend(P_test_4, w_ap4), _tdep))
    rows.append(
        bench_row(
            f"four_grid_pr_divpen_{BLEND_DIVERSITY_LAMBDA:g}_cal",
            blend(P_test_4, w_div4),
            _tdep,
        )
    )
    rows.append(
        bench_row("four_nocat_max_pr_auc_cal", blend(P_test_4, w_nocat_ap4), _tdep)
    )
    rows.append(bench_row("four_nocat_max_f2_cal", blend(P_test_4, w_nocat_f24), _tdep))
    rows.append(
        bench_row("cat_tabpfn_50_50", blend(P_test_4, np.array([0.5, 0.0, 0.0, 0.5])), _tdep)
    )
    rows.append(
        bench_row(
            "xgb_rf_tabpfn_equal_thirds",
            blend(
                P_test_4,
                np.array([0.0, 1.0 / 3, 1.0 / 3, 1.0 / 3]),
            ),
            _tdep,
        )
    )
    rows.append(
        bench_row(
            "three_way_pr_winner_plus_tabpfn_15pct",
            blend(
                P_test_4,
                np.array(
                    [0.85 * w_ap[0], 0.85 * w_ap[1], 0.85 * w_ap[2], 0.15]
                ),
            ),
            _tdep,
        )
    )
    rows.append(
        bench_row(
            "three_way_f2_winner_plus_tabpfn_15pct",
            blend(
                P_test_4,
                np.array(
                    [0.85 * w_f2[0], 0.85 * w_f2[1], 0.85 * w_f2[2], 0.15]
                ),
            ),
            _tdep,
        )
    )
    rows.append(bench_row("four_stacking_lr_on_probs_cal", p_test_stack4, _tdep))
bench = pd.DataFrame(rows)
_bench_cols = [
    "model",
    "roc_auc",
    "pr_auc",
    "precision",
    "recall",
    "f1",
    "accuracy",
    "threshold_deploy",
]
display(bench[[c for c in _bench_cols if c in bench.columns]])
bench.to_csv(os.path.join(RESULT_DIR, "test_ranking_metrics.csv"), index=False)

,model,roc_auc,pr_auc,precision,recall,f1,accuracy,threshold_deploy
0,LR,0.943136,0.473864,0.087500,0.777778,0.157303,0.855352,0.11
1,RF_merge,0.940737,0.599246,0.555556,0.555556,0.555556,0.984571,0.11
2,Cat_cal,0.953822,0.428936,0.302326,0.722222,0.426230,0.966249,0.11
3,XGB_cal,0.971323,0.660000,0.615385,0.444444,0.516129,0.985535,0.11
4,LGB,0.935558,0.475564,0.174419,0.833333,0.288462,0.928640,0.11
5,Ensemble_avg_cal,0.954749,0.679473,0.478261,0.611111,0.536585,0.981678,0.11
6,Ensemble_w_pr_full_cal,0.971323,0.660000,0.615385,0.444444,0.516129,0.985535,0.11
7,Ensemble_w_pr_constrained_min10pct_cal,0.956003,0.647902,0.611111,0.611111,0.611111,0.986500,0.11
8,Ensemble_w_pr_divpen_0.08_cal,0.971323,0.660000,0.615385,0.444444,0.516129,0.985535,0.11
9,Ensemble_w_mean5fold_oof_pr_cal,0.971323,0.660000,0.615385,0.444444,0.516129,0.985535,0.11
